In [3]:
import os
import numpy as np
import pandas as pd
import json
import copy

np.random.seed(1241)

In [4]:
init_dataset = pd.read_json("hf://datasets/Team-ACE/ToolACE/data.json")

/Users/valeria/Desktop/skoltech/DL/dl_project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
init_dataset.head()

,system,conversations
0,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'I'm considering in..."
1,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'Could you please f..."
2,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'Hey, can you show ..."
3,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'Could you provide ..."
4,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'Can you generate a..."


In [9]:
init_dataset['conversations'][0]

[{'from': 'user',
  'value': "I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?"},
 {'from': 'assistant',
  'value': '[Market Trends API(trend_type="MARKET_INDEXES", country="us")]'},
 {'from': 'tool',
  'value': '[{"name": "Market Trends API", "results": {"trends": [{"name": "S&P 500", "description": "Standard & Poor\'s 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J", "description": "Dow Jones Industrial Average is a price-weighted average of 30 blue-chip stocks that are generally the leaders in their industry.", "data": {"current_value": "34479.60", "percentage_change": "+0.47%"}}, {"name": "NASDAQ", "description": "The NASDAQ Composite is a broad-based capitalization-weighted index of stocks in all three NASDAQ tiers: Global Select, Global Market and Cap

In [11]:
init_dataset['system'][0]

'You are an expert in composing functions. You are given a question and a set of possible functions. \nBased on the question, you will need to make one or more function/tool calls to achieve the purpose. \nIf none of the function can be used, point it out. If the given question lacks the parameters required by the function,\nalso point it out. You should only return the function call in tools call sections.\nHere is a list of functions in JSON format that you can invoke:\n[{"name": "newAddress", "description": "Generates a new Ethereum address that can be used to send or receive funds. Do not lose the password! We can\'t restore access to an address if you lose it.", "parameters": {"type": "dict", "properties": {"password": {"description": "The password for the new Ethereum address", "type": "string"}}, "required": ["password"]}, "required": null}, {"name": "Market Trends API", "description": "Get the latest market trends and relevant news for a specified country and language.", "param

## Parser functions

In [5]:
def tool_responses_to_text(responses):
    res = ""
    for response in responses:
        res += f"{response['name']}: {json.dumps(response['results'])}"
    return res

def parse_one_conversation(conv):
    res = []
    N = len(conv)
    i = 0
    while i < N:
        if conv[i]['from'] == 'tool':
            context = copy.deepcopy(json.loads(conv[i]['value']))
            new_entry = {}

            # Looking for the user request
            j = i
            while j >= 0:
                if conv[j]['from'] != 'user':
                    j -= 1
                else:
                    break
            else:
                print("Could not find user request???? Skipping the conversation...")
                return res
            new_entry['query'] = conv[j]['value']

            # Looking for the model response (it is absent in some conversations)
            j = i + 1
            while j < N:
                if conv[j]['from'] != 'assistant':
                    # There are some conversations where tool calls are not combined in one
                    if conv[j]['from'] == 'tool':
                        i = j
                        context += json.loads(conv[j]['value'])
                    j += 1
                else:
                    val = conv[j]['value']
                    if val.startswith('[') and val.endswith(']'): # Means the another tool is called
                        j += 1
                        continue
                    break
            else:
                return res
            new_entry['context'] = tool_responses_to_text(context)
            new_entry['output'] = conv[j]['value']
            res.append(new_entry)
        i += 1
    return res

def extract_tools_list_from_system(system_text: str):
    """
    Extract the *outer* JSON list of tool dicts from a ToolACE system prompt.

    Works even when the JSON contains nested [...] like enums/required, because it
    balances brackets and ignores brackets inside strings.
    """
    # Optional anchor: start searching after the "Here is a list..." line if present
    anchor = "Here is a list of functions"
    start_pos = system_text.find(anchor)
    if start_pos != -1:
        s = system_text[start_pos:]
    else:
        s = system_text

    # Find first '[' (start of the tool list)
    i0 = s.find("[")
    if i0 == -1:
        return None

    depth = 0
    in_str = False
    quote = ""
    esc = False
    start = None

    for i in range(i0, len(s)):
        ch = s[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
            continue

        if ch in ('"', "'"):
            in_str = True
            quote = ch
            continue

        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    block = s[start:i + 1]
                    try:
                        obj = json.loads(block)
                    except Exception:
                        return None

                    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj) and any("name" in x for x in obj):
                        return obj
                    return None

    return None


## Parsing dataset and corrupting it

In [6]:
# List of dicts with keys 'query', 'context', 'output'
correct_dataset = []

for row in init_dataset.itertuples():
    system = row.system
    conv = row.conversations
    parsed_conv = parse_one_conversation(conv)
    parsed_tool_list = extract_tools_list_from_system(system)
    if parsed_tool_list and parsed_conv:
        keys_to_keep = ['name', 'description']
        parsed_tool_list = [{k:v for k, v in curr_tool.items() if k in keys_to_keep} for curr_tool in parsed_tool_list]
        for request in parsed_conv:
            old_context = request['context']
            new_context = f'{old_context}. Available tools: {json.dumps(parsed_tool_list)}.'
            request['context'] = new_context
    correct_dataset += parsed_conv

In [ ]:
def corrupt(dataset, corruption_type, p=0.5):
    """Creates the corrupted dataset out of `dataset`.
    Supported corruption types are 'hallucination', 'overgeneration', 'missing_tool'.
    `p` is the probability of an entry of the `dataset` to be corrupted
    """
    corrupted_dataset = []
    for entry in dataset:
        new_entry = copy.deepcopy(entry)
        if np.random.uniform(0, 1) < p:
            if corruption_type == 'hallucination':
                hallucinate(new_entry)
            elif corruption_type == 'overgeneration':
                overgenerate(new_entry)
            elif corruption_type == 'missing_tool':
                suggest_call_to_missing_tool(new_entry)
            else:
                raise ValueError("Unknown hallucination type")
        else:
            new_entry['hallucination_labels'] = []
        corrupted_dataset.append(new_entry)
    return corrupted_dataset

def hallucinate(entry):
    raise NotImplementedError()

def overgenerate(entry):
    raise NotImplementedError()

def suggest_call_to_missing_tool(entry):
    raise NotImplementedError()

In [ ]:
hallucinated_dataset = corrupt(correct_dataset, 'hallucination', 0.0)
hallucinated_dataset[:2]

[{'query': "I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?",
  'context': 'Market Trends API: {"trends": [{"name": "S&P 500", "description": "Standard & Poor\'s 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J", "description": "Dow Jones Industrial Average is a price-weighted average of 30 blue-chip stocks that are generally the leaders in their industry.", "data": {"current_value": "34479.60", "percentage_change": "+0.47%"}}, {"name": "NASDAQ", "description": "The NASDAQ Composite is a broad-based capitalization-weighted index of stocks in all three NASDAQ tiers: Global Select, Global Market and Capital Market.", "data": {"current_value": "13691.30", "percentage_change": "+0.90%"}}]}. Available tools: [{"name": "newAddress", "description": "Generates a ne

In [7]:
correct_dataset[:2]

[{'query': "I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?",
  'context': 'Market Trends API: {"trends": [{"name": "S&P 500", "description": "Standard & Poor\'s 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J", "description": "Dow Jones Industrial Average is a price-weighted average of 30 blue-chip stocks that are generally the leaders in their industry.", "data": {"current_value": "34479.60", "percentage_change": "+0.47%"}}, {"name": "NASDAQ", "description": "The NASDAQ Composite is a broad-based capitalization-weighted index of stocks in all three NASDAQ tiers: Global Select, Global Market and Capital Market.", "data": {"current_value": "13691.30", "percentage_change": "+0.90%"}}]}. Available tools: [{"name": "newAddress", "description": "Generates a ne